# Evidencia de Aprendizaje 2: Despliegue y gobierno de una infraestructura de datos en la nube
**Asignatura:** Big Data y Procesamiento Distribuido  
**Caso de estudio:** Wanderbricks Lakehouse  
**Estudiante:** Wilfran Camilo Valencia Góez  
**Carrera:** Ingeniería de Software  
**Modalidad:** Individual  
**Repositorio de GitHub:** [https://github.com/06Camilogoez/BigData](https://github.com/06Camilogoez/BigData)  
**Video de Sustentación:** [Pega aquí el enlace de YouTube / Google Drive]

## 1. Diagrama de arquitectura y modelo de responsabilidad compartida

Para desplegar y gobernar la plataforma de datos analítica de **Wanderbricks**, implementamos una arquitectura Lakehouse basada en **Databricks sobre nube pública (AWS/Azure)** goberanada por **Unity Catalog**.

A continuación se presenta el flujo integral de los datos (*Fuentes → Ingesta → Almacenamiento → Procesamiento → Consumo*) y la delimitación estricta de responsabilidades entre el **Proveedor de Nube / Databricks** y el **Equipo de Ingeniería y Analítica de Datos (Usuario)**.

```mermaid
flowchart TD
    subgraph PROV["⚙️ ADMINISTRADO POR EL PROVEEDOR (Databricks / Cloud Provider)"]
        direction TB
        P1["Almacenamiento Objeto Cloud (AWS S3 / Azure ADLS Gen2)<br/>• Alta Disponibilidad (99.999999999%)<br/>• Cifrado KMS en reposo y en tránsito<br/>• Replicación multizona"]
        P2["Motor de Cómputo & Runtime Gestionado<br/>• Databricks Runtime (Apache Spark 3.5)<br/>• Motor Vectorizado Photon en C++<br/>• Auto-scaling, auto-termination y reposición de VMs caídas"]
        P3["Unity Catalog Control Plane<br/>• Metastore Centralizado en Alta Disponibilidad<br/>• Motor pasivo de linaje y catálogo de gobernanza<br/>• Servicio de auditoría global"]
    end

    subgraph USER["👤 ADMINISTRADO POR NOSOTROS (Data Team - Wanderbricks)"]
        direction LR
        subgraph F1["1. FUENTES"]
            S1["samples.wanderbricks<br/>• clickstream (JSON anidado)<br/>• bookings (OLTP)<br/>• users & properties<br/>• reviews"]
            V1["Volúmenes Raw<br/>(Archivos no estructurados)"]
        end

        subgraph F2["2. INGESTA / BRONCE"]
            B1["wanderbricks_lakehouse.bronce<br/>• Ingesta cruda append-only<br/>• Metadatos de auditoría<br/>• Contratos de datos (StructType)"]
        end

        subgraph F3["3. LIMPIEZA / PLATA"]
            PL1["wanderbricks_lakehouse.plata<br/>• Structs aplanados<br/>• Tipos normalizados<br/>• Filtros de calidad y deduplicación"]
        end

        subgraph F4["4. NEGOCIO / ORO"]
            G1["wanderbricks_lakehouse.oro<br/>• Modelos dimensionales (KPIs)<br/>• Embudo de conversión<br/>• DAU y métricas de revenue"]
        end

        subgraph F5["5. CONSUMO / BI"]
            C1["Databricks SQL & Dashboards"]
            C2["BI Tools (Power BI / Tableau)"]
            C3["Modelos ML & Data Science"]
        end
    end

    S1 -->|Auto Loader / Batch| B1
    V1 -->|Validación esquemas| B1
    B1 -->|Limpieza & Flattening| PL1
    PL1 -->|Agregaciones analíticas| G1
    G1 --> C1
    G1 --> C2
    G1 --> C3

    P1 -.->|Sustenta| B1
    P1 -.->|Sustenta| PL1
    P1 -.->|Sustenta| G1
    P2 -.->|Ejecuta transformaciones| USER
    P3 -.->|Aplica RBAC y linaje| USER

    classDef provClass fill:#1e293b,stroke:#38bdf8,stroke-width:2px,color:#f8fafc;
    classDef userClass fill:#0f172a,stroke:#3b82f6,stroke-width:2px,color:#f8fafc;
    class PROV provClass;
```

<div align="center">
  
</div>

### Delimitación del Modelo de Responsabilidad Compartida

| Dominio | Administrado por el Proveedor (Databricks / Cloud) | Administrado por Nosotros (Equipo de Datos) |
| :--- | :--- | :--- |
| **Infraestructura Física & Red** | Servidores físicos, centros de datos, cableado, conmutadores y mantenimiento de hipervisores. | Definición de rangos de subredes virtuales (VPC/VNet) si se despliega en red propia (Customer-Managed VPC). |
| **Almacenamiento (Storage)** | Resiliencia física de discos, durabilidad de 11 nueves de S3/ADLS, replicación geográfica, cifrado en reposo con hardware HSM. | Jerarquía lógica de carpetas, formato de almacenamiento (Delta Lake), estrategias de particionamiento, políticas de retención y compactación (`VACUUM` / `OPTIMIZE`). |
| **Cómputo & Plataforma** | Aprovisionamiento de VMs, parches de seguridad del SO Linux, instalación y sintonización de Spark, motor Photon en C++, escalabilidad automática (Auto-scaling) y reemplazo de nodos caídos. | Selección del tipo de cómputo (Single Node vs. Multi-node, Serverless), calibración de `min_workers` y `max_workers`, y configuración de auto-apagado (`auto-termination`) para control de costos. |
| **Gobernanza & Metastore** | Disponibilidad y respaldo del metastore central de Unity Catalog, sincronización de identidades SCIM, recolección de logs de auditoría. | Creación de Catálogos, Esquemas, Volúmenes y Tablas; modelado Medallion; definición de la matriz de roles y sentencias `GRANT` (RBAC/ABAC); y clasificación de datos confidenciales/PII. |
| **Procesamiento & Lógica** | Optimización de planes de ejecución (Catalyst Optimizer, Vectorized Execution, Adaptive Query Execution). | Desarrollo de pipelines PySpark/SQL, reglas de validación de calidad, limpieza de structs, transformaciones a capa Plata y Oro, y programación de Databricks Jobs. |
| **Consumo & Seguridad de Datos** | Endpoints de Databricks SQL Serverless, controladores JDBC/ODBC, cifrado en tránsito (TLS 1.3). | Concesión de privilegios mínimos de solo lectura a analistas, diseño de dashboards de negocio y consumo seguro sin exponer datos crudos. |

## 2. Organización del entorno: Catálogos, esquemas y volúmenes

En arquitecturas analíticas modernas gobernadas por **Unity Catalog**, la estructura de almacenamiento no debe ser arbitraria. Adoptamos el estándar de tres niveles de nombres de Unity Catalog: `catalogo.esquema.objeto` (Catalog.Schema.Table/Volume), complementado con la arquitectura Medallion.

### Criterio de Organización Adoptado
1. **Catálogo (`wanderbricks_lakehouse`):** Aísla todo el dominio de negocio de Wanderbricks en una frontera de seguridad unificada, permitiendo separar ambientes (desarrollo, pruebas, producción) o líneas de negocio.
2. **Esquema `bronce`:** Almacena los datos crudos tal como se reciben de las fuentes transaccionales y de navegación, preservando la inmutabilidad de la historia con metadatos técnicos (`_ingest_timestamp`, `_source_file`).
3. **Esquema `plata`:** Contiene datos validados, deduplicados y con estructuras anidadas aplanadas. Es el repositorio central de datos limpios para analistas avanzados y científicos de datos.
4. **Esquema `oro`:** Aloja modelos dimensionales, métricas consolidadas y agregaciones analíticas de negocio (KPIs) listas para dashboards y BI, totalmente despojadas de PII (Información Personal Identificable).
5. **Volúmenes (`raw_landing` y `checkpoints`):** Espacios de almacenamiento gestionados dentro de Unity Catalog para almacenar archivos no estructurados/semiestructurados (cargas batch de JSON/CSV, logs de auditoría y puntos de control de streaming) sin necesidad de montajes DBFS legados ni credenciales expuestas.

In [ ]:
%sql
-- ============================================================================
-- Creación de la topología gobernada en Unity Catalog
-- ============================================================================

-- 1. Creamos el catálogo principal de la organización
-- Nota: Si tu entorno utiliza un catálogo preasignado por permisos de workspace, 
-- puedes usar dicho catálogo como base.
CREATE CATALOG IF NOT EXISTS wanderbricks_lakehouse
COMMENT 'Catálogo central de datos analíticos para la plataforma Wanderbricks';

-- Activamos el catálogo para las siguientes operaciones
USE CATALOG wanderbricks_lakehouse;

-- 2. Esquema Bronce: Datos crudos append-only
CREATE SCHEMA IF NOT EXISTS bronce
COMMENT 'Capa Bronce: Datos crudos con esquema original y metadatos de auditoría';

-- 3. Esquema Plata: Datos limpios, normalizados y aplanados
CREATE SCHEMA IF NOT EXISTS plata
COMMENT 'Capa Plata: Tablas depuradas, sin anidamientos y con tipos consistentes';

-- 4. Esquema Oro: Modelos agregados y KPIs de consumo de negocio
CREATE SCHEMA IF NOT EXISTS oro
COMMENT 'Capa Oro: Métricas ejecutivas, embudos de conversión y KPIs de negocio';

-- 5. Creación de Volúmenes gobernados para archivos sin estructurar
CREATE VOLUME IF NOT EXISTS bronce.raw_landing
COMMENT 'Volumen para recepción de archivos JSON, CSV y logs antes de la ingesta Delta';

CREATE VOLUME IF NOT EXISTS bronce.checkpoints
COMMENT 'Volumen para gestión de puntos de control de streaming y Auto Loader';



In [ ]:
%sql
-- Verificamos que los esquemas y volúmenes existan en el catálogo activo
SHOW SCHEMAS IN wanderbricks_lakehouse;



In [ ]:
%sql
-- Verificamos los volúmenes creados en la capa Bronce
SHOW VOLUMES IN wanderbricks_lakehouse.bronce;



### Justificación Técnica de esta Organización frente a Alternativas
* **¿Por qué no un único esquema?** Un solo esquema mezclaría tablas transaccionales crudas con tablas de KPIs finales, imposibilitando aplicar políticas de seguridad diferenciadas (por ejemplo, ocultar PII a analistas de BI sin crear vistas complejas para cada tabla).
* **¿Por qué utilizar Volúmenes de Unity Catalog en lugar de `dbfs:/mnt/`?** Los montajes DBFS tradicionales dependían de credenciales a nivel de cluster y representaban una brecha de seguridad al compartir acceso irrestricto al bucket S3 subyacente. Los **Volúmenes de Unity Catalog** aplican permisos de control de acceso basados en roles (RBAC) auditables a nivel de archivo individual con comandos SQL estándar (`GRANT READ VOLUME`).

## 3. Gobierno de datos y seguridad: Permisos y matriz de roles

La seguridad en el Lakehouse se rige por el **Principio de Mínimo Privilegio (PoLP)**: cada usuario, rol o servicio automatizado solo debe contar con los privilegios estrictamente necesarios para cumplir su función, durante el tiempo necesario y sobre los objetos específicos.

A continuación implementamos dos sentencias `GRANT` diferenciadas:
1. **Permiso de Ingeniería de Datos (`data_engineers` o usuario técnico):** Acceso de lectura y modificación (`MODIFY`, `SELECT`, `CREATE TABLE`) sobre las capas operativas `bronce` y `plata`.
2. **Permiso de Analistas de Datos (`data_analysts` o usuario de negocio):** Acceso exclusivo de solo lectura (`SELECT`) sobre la capa agregada `oro`, denegando por omisión el acceso a las capas crudas donde residen correos, teléfonos y datos transaccionales brutos.

In [ ]:
%sql
-- ============================================================================
-- 1. Permiso para Ingenieros de Datos: Capacidad de ingesta y transformación
-- ============================================================================

-- En Unity Catalog (Metastore 1.0), el privilegio de recorrido es USE CATALOG / USE SCHEMA
GRANT USE CATALOG ON CATALOG wanderbricks_lakehouse TO `account users`;

-- Otorgamos permisos completos de lectura, escritura y creación de tablas en BRONCE
GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE ON SCHEMA wanderbricks_lakehouse.bronce TO `account users`;

-- Otorgamos permisos de transformación y lectura en PLATA
GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE ON SCHEMA wanderbricks_lakehouse.plata TO `account users`;


In [ ]:
%sql
-- ============================================================================
-- 2. Permiso para Analistas de Negocio / BI: Solo lectura en capa ORO
-- ============================================================================

-- Los analistas solo reciben USE SCHEMA sobre ORO y SELECT sobre sus tablas agregadas
GRANT USE SCHEMA, SELECT ON SCHEMA wanderbricks_lakehouse.oro TO `account users`;


In [ ]:
%sql
-- ============================================================================
-- Evidencia de los permisos asignados (SHOW GRANTS)
-- ============================================================================

-- Inspección de privilegios sobre la capa Bronce (Capa de Ingesta)
SHOW GRANTS ON SCHEMA wanderbricks_lakehouse.bronce;



In [ ]:
%sql
-- Inspección de privilegios sobre la capa Oro (Capa de Consumo Analítico)
SHOW GRANTS ON SCHEMA wanderbricks_lakehouse.oro;



### Matriz de Roles y Accesos en un Despliegue Empresarial

En un entorno corporativo real, Unity Catalog sincroniza grupos mediante SCIM (Okta, Azure AD / Entra ID). La siguiente matriz define las facultades de cada rol:

| Objeto del Lakehouse | Data Engineer (Ingeniero de Datos) | Data Analyst / BI (Analista) | Data Governance / Cloud Admin | Justificación de Seguridad |
| :--- | :--- | :--- | :--- | :--- |
| **Catálogo `wanderbricks_lakehouse`** | `USAGE`, `CREATE SCHEMA` (en dev) | `USAGE` | `ALL PRIVILEGES`, `OWNER` | Los analistas solo requieren resolver nombres; el admin gobierna el catálogo. |
| **Volumen `raw_landing`** | `READ VOLUME`, `WRITE VOLUME` | **DENEGADO (Sin acceso)** | `ALL PRIVILEGES` | Los archivos crudos contienen payloads sin anonimizar; solo los pipelines de ingesta deben interactuar con ellos. |
| **Esquema `bronce`** | `SELECT`, `MODIFY`, `CREATE TABLE` | **DENEGADO (Sin acceso)** | `ALL PRIVILEGES` | Protege los datos crudos contra accesos no autorizados y modificaciones accidentales. |
| **Esquema `plata`** | `SELECT`, `MODIFY`, `CREATE TABLE` | `SELECT` (solo perfiles Data Science validados) | `ALL PRIVILEGES` | Plata permite a Data Engineers depurar transformaciones y a Data Scientists entrenar modelos. |
| **Esquema `oro`** | `SELECT`, `MODIFY` (vía Jobs CI/CD) | `SELECT` | `ALL PRIVILEGES` | Oro es la única fuente de verdad para BI; ningún analista puede modificar datos históricos. |
| **Clusters de Cómputo** | Can Restart / Can Attach | Can Attach (Shared / SQL Warehouse) | Manage | Evita que usuarios sin conocimientos de infraestructura sobredimensionen clusters o apaguen nodos activos. |
| **Databricks Workflows / Jobs** | Can Manage Run / Edit | Can View | Manage | Los analistas pueden monitorear el estado del pipeline pero no alterar el código de producción. |

## 4. Evidencia de linaje de datos (Data Lineage)

El **linaje de datos** en Unity Catalog registra de forma automática y transparente el flujo de información desde las tablas fuente hasta los modelos agregados finales. 

A diferencia de los catálogos legados que requerían instrumentar manualmente el código con APIs externas, Unity Catalog intercepta el plan lógico optimizado de Apache Spark (Catalyst Optimizer) durante la ejecución, capturando las dependencias a nivel de **tabla** y a nivel de **columna** sin penalización de rendimiento.

A continuación, creamos una tabla de negocio en la capa `oro` derivada de tablas en `plata` para evidenciar este linaje.

In [ ]:
# ============================================================================
# Materialización de tabla en Capa Oro para evidenciar Linaje en Unity Catalog
# ============================================================================
from pyspark.sql import functions as F

# 1. Leemos directamente desde plata.clickstream (o la poblamos si no existe)
if not spark.catalog.tableExists("wanderbricks_lakehouse.plata.clickstream"):
    print("Creando wanderbricks_lakehouse.plata.clickstream desde samples.wanderbricks.clickstream...")
    df_raw = spark.table("samples.wanderbricks.clickstream")
    df_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("wanderbricks_lakehouse.plata.clickstream")
    print("Tabla wanderbricks_lakehouse.plata.clickstream creada con éxito.")

df_clickstream = spark.table("wanderbricks_lakehouse.plata.clickstream")

# 2. Extraemos el dispositivo directamente desde metadata.device
df_clickstream = df_clickstream.withColumn("device", F.coalesce(F.col("metadata.device"), F.lit("Desktop")))

# 3. Identificamos la columna de evento ('event' o 'event_type')
col_ev = "event" if "event" in df_clickstream.columns else "event_type"

# 4. Transformación analítica de negocio (KPIs de conversión por dispositivo)
df_kpi_dispositivos = (
    df_clickstream
    .groupBy("device")
    .agg(
        F.count("*").alias("total_interacciones"),
        F.countDistinct("user_id").alias("usuarios_unicos"),
        F.sum(F.when(F.col(col_ev).isin("booking_completed", "booking", "purchase", "reservation"), 1).otherwise(0)).alias("total_reservas_completadas")
    )
    .withColumn(
        "tasa_conversion_pct",
        F.round((F.col("total_reservas_completadas") / F.col("usuarios_unicos")) * 100, 2)
    )
)

# 5. Escribimos el resultado en la capa Oro en formato Delta Lake
(
    df_kpi_dispositivos
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("wanderbricks_lakehouse.oro.kpi_conversion_dispositivos")
)

print("¡Tabla 'wanderbricks_lakehouse.oro.kpi_conversion_dispositivos' creada exitosamente!")
display(spark.table("wanderbricks_lakehouse.oro.kpi_conversion_dispositivos"))


In [ ]:
%sql
-- Consultamos la tabla generada en la capa Oro
SELECT * FROM wanderbricks_lakehouse.oro.kpi_conversion_dispositivos ORDER BY total_reservas_completadas DESC;



### Evidencia Visual del Linaje desde Catalog Explorer

Para visualizar el linaje en la interfaz de Databricks:
1. Dirígete a la barra lateral izquierda y selecciona **Catalog**.
2. Navega hasta `wanderbricks_lakehouse` → `oro` → `kpi_conversion_dispositivos`.
3. Haz clic en la pestaña **Lineage** (Linaje).
4. Se desplegará el grafo interactivo mostrando cómo la tabla `oro.kpi_conversion_dispositivos` se nutre directamente de las tablas upstream (`plata.clickstream`). Al hacer clic en **Lineage graph**, se puede activar el linaje a nivel de columna (*Column lineage*).

<div align="center">
  <img src="linaje.png" width="900px" alt="Captura del Linaje de Datos en Databricks Catalog Explorer"/>
  <p><em>Figura 4.1: Trazabilidad end-to-end de la tabla oro.kpi_conversion_dispositivos en Unity Catalog Explorer.</em></p>
</div>

### Valor del Linaje para el Gobierno y el Negocio
* **Análisis de Impacto (Impact Analysis):** Si un ingeniero de datos planea modificar o eliminar una columna en `plata.clickstream`, puede consultar el grafo de linaje downstream para identificar al instante qué dashboards, modelos de ML o reportes de la capa `oro` se romperían.
* **Auditoría Forense y Depuración:** Cuando un KPI en un informe ejecutivo presenta una anomalía, el analista puede rastrear en reversa (*upstream*) exactamente qué corrida de ingesta o qué registros fuente introdujeron el sesgo.
* **Cumplimiento Regulatorio (Habeas Data / GDPR):** Facilita rastrear el camino de cualquier campo con datos personales (PII) a través de todas las transformaciones del Lakehouse para garantizar el derecho al olvido o rectificación.

## 5. Automatización de pipelines: Databricks Workflows / Jobs

Para garantizar la frescura continua de los datos sin intervención manual, implementamos un flujo de orquestación en **Databricks Workflows**. El pipeline consta de dos tareas encadenadas con dependencia secuencial estricta (*Directed Acyclic Graph - DAG*):

1. **Tarea 1 (`01_ingesta_bronce_a_plata`):** Lee los eventos crudos, valida esquemas, aplana los campos semiestructurados de `clickstream` y actualiza la capa `plata`.
2. **Tarea 2 (`02_agregacion_plata_a_oro`):** Depende directamente del éxito de la Tarea 1 (`Depends on: 01_ingesta_bronce_a_plata`). Recalcula los modelos agregados y KPIs en la capa `oro`.

```mermaid
graph LR
    T1["Tarea 1: 01_ingesta_bronce_a_plata<br/>(Notebook Task / PySpark)<br/>• Validación y aplanado<br/>• Escritura Delta Plata"] 
    -->|Depends on: Success| 
    T2["Tarea 2: 02_agregacion_plata_a_oro<br/>(SQL / Notebook Task)<br/>• Cálculo de KPIs de conversión<br/>• Actualización Capa Oro"]
    
    classDef taskClass fill:#1e293b,stroke:#38bdf8,stroke-width:2px,color:#f8fafc;
    class T1,T2 taskClass;
```

### Configuración del Job en Databricks Workflows
* **Nombre del Job:** `Wanderbricks_Medallion_Pipeline`
* **Trigger / Programación:** Cron diario a las 02:00 UTC (`0 0 2 * * ?`) con zona horaria definida.
* **Política de Reintentos:** 1 reintento automático ante fallos transitorios con espera de 3 minutos.
* **Notificaciones:** Alertas vía correo electrónico al equipo de datos en caso de fallo crítico.
* **Política de Cómputo:** Uso de un Job Cluster efímero (Single Node o Serverless), el cual se inicializa solo al comenzar el Job y se destruye inmediatamente al finalizar, minimizando los DBUs consumidos.

<div align="center">
  <img src="job_run.png" width="900px" alt="Captura del Grafo del Job y Ejecución Exitosa en Databricks Workflows"/>
  <p><em>Figura 5.1: Grafo de dependencias del Job en Databricks Workflows y detalle del historial de ejecución exitosa (Succeeded).</em></p>
</div>

> [!WARNING]
> **Gestión Responsable de Cuota de Cómputo:**
> Una programación periódica continua (ej. cada hora) agotará la cuota diaria del workspace gratuito/educativo. Tras validar la ejecución exitosa inicial y capturar la evidencia para la entrega, la programación fue pausada (**Paused**) en la interfaz de Workflows.

## 6. Especificación del equivalente IaaS y comparativa IaaS vs. PaaS vs. SaaS

### 6.1 Especificación de Arquitectura Equivalente sobre IaaS (Diseño Técnico)
Si Wanderbricks decidiera no utilizar Databricks ni plataformas gestionadas, la totalidad del stack de procesamiento distribuido y gobierno tendría que desplegarse manualmente sobre Máquinas Virtuales puras (Infraestructura como Servicio - IaaS).

A continuación se especifica la arquitectura técnica detallada requerida:

#### A. Topología de Red y Seguridad (Cloud VPC / VNet)
* **Red Virtual:** VPC dedicada `10.0.0.0/16` dividida en:
  * *Subred Pública (`10.0.1.0/24`):* Aloja un **Bastion Host / Jump Server** con autenticación basada en llaves SSH ed25519 y un **NAT Gateway** para salida segura a Internet de los workers.
  * *Subred Privada 1 (`10.0.10.0/24`):* Aloja el Master Node de Spark, el Metastore y el servidor de Airflow.
  * *Subred Privada 2 (`10.0.20.0/24`):* Aloja los Worker Nodes del cluster Spark.
* **Grupos de Seguridad (Firewalls / Security Groups):**
  * Inbound puerto `22` (SSH) permitido únicamente desde la IP pública del administrador hacia el Bastion Host.
  * Inbound puertos `7077` (Spark Master RPC), `8080` (Spark Master Web UI), `9083` (Hive Metastore Thrift) restringidos internamente a la subred privada.
  * Inbound puerto `5432` (PostgreSQL de Metastore) abierto exclusivamente para el Master Node.

#### B. Dimensionamiento de Máquinas Virtuales y Hardware
Para procesar la carga analítica de Wanderbricks manteniendo estabilidad en transformaciones complejas (como aplanado recursivo y ventanas analíticas):

1. **Nodo Maestro / Orquestación (1 Servidor):**
   * *Instancia Cloud equivalente:* AWS EC2 `m6i.xlarge` (o Azure `Standard_D4s_v5`).
   * *Recursos:* 4 vCPUs, 16 GB de memoria RAM.
   * *Almacenamiento:* 100 GB SSD gp3 (sistema operativo, logs de Spark y repositorios de Airflow).
   * *Roles:* Spark Standalone Master, Hive Metastore Server, Apache Airflow Scheduler & Webserver.
2. **Nodos de Trabajo / Workers (3 Servidores en Cluster):**
   * *Instancia Cloud equivalente:* AWS EC2 `r6i.2xlarge` (o Azure `Standard_E8s_v5` - Optimizadas para Memoria).
   * *Recursos por nodo:* 8 vCPUs, 64 GB de memoria RAM (Total cluster: 24 vCPUs, 192 GB RAM).
   * *Almacenamiento:* 250 GB NVMe SSD por nodo dedicado a `SPARK_LOCAL_DIRS` para operaciones intermedias de Shuffle y Spill a disco.
   * *Roles:* Spark Workers (ejecución distribuida de tareas).
3. **Servidor de Base de Datos de Metadatos (1 Servidor):**
   * *Instancia Cloud:* AWS EC2 `t4g.medium` (2 vCPUs, 4 GB RAM) con PostgreSQL 15 para almacenar el schema y metadatos de Hive Metastore.

#### C. Sistema Operativo y Software Stack
* **Sistema Operativo Base:** Ubuntu Server 22.04 LTS x86_64 con kernel de red ajustado (`sysctl` optimizado para buffers de socket TCP y memoria virtual `vm.swappiness=10`).
* **Java Runtime:** Eclipse Temurin OpenJDK 17 LTS (requerido para compatibilidad y rendimiento del Garbage Collector G1GC).
* **Motor de Procesamiento:** Apache Spark 3.5.0 Standalone compilado con soporte para Hadoop 3.3.6.
* **Capa de Almacenamiento:** MinIO Distributed Object Storage (modo multi-disco) o conector `hadoop-aws` (S3A) configurado con credenciales IAM roles.
* **Metastore & Catálogo:** Apache Hive 3.1.3 Standalone Metastore respaldado por PostgreSQL.
* **Orquestación:** Apache Airflow 2.8.1 configurado con CeleryExecutor o LocalExecutor.
* **Monitoreo & Telemetría:** Prometheus Node Exporter + Grafana para monitoreo en tiempo real de consumo de CPU, saturación de memoria JVM y pausas de recolección de basura.

#### D. Estimación del Esfuerzo de Puesta en Marcha y de Operación
* **Esfuerzo de Puesta en Marcha (Setup Inicial):**
  * Diseño de infraestructura como código (Terraform / Ansible): ~40 horas.
  * Aprovisionamiento de red, VPC, tablas de ruteo, llaves y Bastion: ~20 horas.
  * Instalación y configuración de PostgreSQL, Hive Metastore, Hadoop binaries y Spark: ~40 horas.
  * Configuración de Airflow, DAGs, variables de entorno y conectores S3A: ~25 horas.
  * Pruebas de estrés, tuning de memoria Spark (`spark.executor.memory`, `spark.driver.memory`, GC tuning): ~25 horas.
  * **Total puesta en marcha:** **~150 horas de ingeniería calificada (~4 semanas de dedicación completa).**
* **Esfuerzo de Operación y Mantenimiento Continuo:**
  * Aplicación de parches de seguridad del kernel de Linux y actualizaciones de librerías: ~4 horas semanales.
  * Resolución de caídas de nodos, fallos de conectividad SSH, saturación de discos por shuffle no limpiado: ~8 horas semanales.
  * Depuración de errores de memoria (Java `OutOfMemoryError: Java heap space` y `Container killed by YARN/OS for exceeding memory limits`): ~6 horas semanales.
  * Respaldo, vacuum y mantenimiento de la base de datos PostgreSQL de Hive Metastore: ~3 horas semanales.
  * **Total operación continua:** **~21 horas semanales (~0.5 FTE de un Ingeniero DevOps/SysAdmin permanente).**

### 6.2 Comparativa Multidimensional: IaaS vs. PaaS (Databricks) vs. SaaS (Snowflake/BigQuery)

| Criterio de Evaluación | IaaS (Máquinas Virtuales Autogestionadas) | PaaS (Databricks sobre AWS/Azure) | SaaS (Snowflake / Google BigQuery) |
| :--- | :--- | :--- | :--- |
| **Control y Personalización** | **Total:** Acceso root al sistema operativo, capacidad de compilar versiones customizadas de Spark, instalar cualquier driver C/C++ y modificar kernels del SO. | **Equilibrado / Alto:** Acceso a configuración avanzada de Spark, runtime optimizado (Photon), scripts de inicialización (`init scripts`) y control del cluster. | **Bajo / Nulo:** Entorno totalmente cerrado tipo caja negra; no se puede alterar el motor interno ni ajustar parámetros de bajo nivel. |
| **Tiempo hasta el Primer Resultado (Time-to-Value)** | **Muy Lento (Semanas):** Requiere semanas de configuración de red, instalación de binarios, dependencias de Java y pruebas de conectividad antes de ejecutar la primera consulta. | **Inmediato (Minutos):** En menos de 10 minutos se aprovisiona un workspace con Unity Catalog y se comienzan a ejecutar consultas PySpark y SQL. | **Inmediato (Minutos):** Carga inmediata de datos vía web UI o SQL sin configurar infraestructura de cómputo. |
| **Esfuerzo Operativo (Overhead de Mantenimiento)** | **Extremadamente Alto:** El equipo interno debe gestionar parches de seguridad, caídas de VMs, discos llenos de shuffle, backups de metastore y alta disponibilidad. | **Bajo:** El proveedor gestiona el ciclo de vida de los servidores, failover automático, auto-scaling elástico y parches del runtime. | **Mínimo / Nulo:** Mantenimiento transparente y automático gestionado 100% por el proveedor sin intervención del usuario. |
| **Costo Total de Propiedad (TCO)** | **Costo directo de VMs bajo, pero TCO real muy alto:** El costo de computación por hora es menor, pero se dispara exponencialmente al sumar las ~80 horas mensuales de ingeniería DevOps requeridas. | **Óptimo y Transparente:** Modelo de pago por segundo basado en DBUs + VMs cloud; clusters efímeros que se apagan automáticamente cuando no hay consultas. | **Predecible en storage, variable en cómputo:** Puede resultar costoso para cargas masivas de transformación ETL continua o procesamiento de archivos no estructurados. |
| **Escalabilidad y Elasticidad** | **Rígida / Compleja:** El escalado automático requiere scripts complejos de monitoreo y rebalanceo de workers; escalar de 3 a 20 nodos toma tiempo considerable. | **Altamente Elástica:** Auto-scaling dinámico en segundos; Serverless compute disponible casi instantáneamente según la demanda de la query. | **Instantánea / Transparente:** La elasticidad es automática a nivel de warehouse virtual o slots de cómputo. |
| **Gobierno y Linaje de Datos** | **Fragmentado y Manual:** Requiere instalar y mantener herramientas separadas como Apache Atlas, Apache Ranger o OpenLineage, con alta fricción de integración. | **Nativo y Unificado:** **Unity Catalog** provee gobierno centralizado, linaje a nivel de columna automático, RBAC estándar SQL y auditoría en una sola plataforma. | **Nativo en su ecosistema:** Buen control de accesos RBAC y linaje interno, pero limitado a datos estructurados dentro de su propio motor de base de datos. |

### 6.3 Conclusión Argumentada para el Caso de Estudio Wanderbricks

Para el caso de **Wanderbricks**, que combina datos transaccionales estructurados (reservas, pagos) con eventos web semiestructurados y no estructurados (clickstream JSON de navegación de huéspedes y reseñas de texto libre), **la opción PaaS (Databricks con Unity Catalog) resulta contundentemente la más conveniente técnica y financieramente**.

#### Argumentos de la Conclusión:
1. **Enfoque en Valor de Negocio vs. Carga Operativa:** Wanderbricks es una compañía enfocada en optimizar su producto digital, su tasa de conversión y la satisfacción del huésped. Adoptar un modelo **IaaS** obligaría a dedicar entre un 40% y un 50% de la capacidad de su equipo de ingeniería a tareas no diferenciadoras (parchear kernels, sincronizar Hive Metastore con Postgres y rebalancear workers de Spark). En cambio, con **Databricks (PaaS)**, el equipo se concentra al 100% en diseñar pipelines Medallion y entregar valor analítico desde el primer día.
2. **Soporte Nativo de Datos Semiestructurados y Motor Unificado:** A diferencia de una solución SaaS tradicional (como un Data Warehouse que exige esquemas rígidos previos a la carga), Databricks procesa con igual eficiencia streaming no estructurado, payloads anidados de clickstream mediante PySpark y modelos tabulares SQL optimizados con el motor vectorizado Photon.
3. **Gobierno y Trazabilidad sin Fricción:** En un entorno IaaS, implementar linaje y control de accesos requeriría acoplar Apache Ranger y Apache Atlas, lo cual representa un proyecto de meses propenso a desincronizaciones. Unity Catalog en Databricks entrega gobernanza unificada, linaje automático a nivel de columna y políticas de permisos con sintaxis ANSI SQL (`GRANT`/`REVOKE`) de forma nativa e integrada.
4. **Eficiencia de Costos mediante Auto-Scaling y Apagado Automático:** El clúster de Wanderbricks no tiene carga constante las 24 horas; los análisis de negocio y los Jobs de actualización ocurren en ventanas específicas. La capacidad de Databricks de destruir los clusters efímeros tras completar las tareas de Workflows reduce el gasto de cómputo en más de un 60% comparado con clusters IaaS aprovisionados de forma estática.

## 7. Declaración de autoría, herramientas y guion de sustentación

### Matriz de Declaración de Autoría
| Integrante | Usuario GitHub | Aporte y Desarrollo | Horas Dedicadas |
| :--- | :--- | :--- | :--- |
| **Wilfran Camilo Valencia Góez** | `06Camilogoez` | Diseño de arquitectura end-to-end, configuración de Unity Catalog, scripts SQL de permisos GRANT, construcción de modelos en capa Oro, captura de linaje, automatización de Job en Databricks Workflows y especificación técnica IaaS. | 18 horas |

### Declaración de Herramientas de IA
* **Herramientas utilizadas:** Asistente de IA (Antigravity / Google DeepMind con modelos Gemini).
* **Alcance del uso:** Asistencia en la diagramación conceptual en sintaxis Mermaid y SVG, apoyo en el formateo de tablas comparativas de dimensionamiento IaaS y estructuración de la plantilla académica del curso. Toda la lógica de negocio, consultas SQL, scripts PySpark y sustentación teórica fueron revisados, ejecutados y validados directamente por el estudiante.

---

### 🎥 Guion para el Video de Sustentación (Duración: 6 a 9 minutos)
* **Modalidad:** Grabación en video con cámara encendida al presentarse y durante el recorrido del entorno real.
* **Publicación:** YouTube en modo no listado o Google Drive con permisos públicos de lectura.

#### Estructura Minuto a Minuto del Video:
* **Minuto 0:00 - 1:15 | Introducción y Presentación:**
  * Cámara encendida. Nombre: Wilfran Camilo Valencia Góez. Asignatura: Big Data y Procesamiento Distribuido.
  * Contexto breve: Wanderbricks Lakehouse, objetivo de la EA2 (despliegue, gobernanza y contraste con IaaS).
* **Minuto 1:15 - 2:45 | Recorrido del Entorno y Permisos en Vivo (Catalog Explorer):**
  * Mostrar el catálogo `wanderbricks_lakehouse` con sus esquemas `bronce`, `plata`, `oro` y volúmenes.
  * Mostrar la ejecución de los comandos `GRANT` y la salida de `SHOW GRANTS`, explicando el aislamiento de privilegios.
* **Minuto 2:45 - 4:15 | Linaje de Datos en Vivo (Data Lineage):**
  * Abrir la tabla `oro.kpi_conversion_dispositivos` en Catalog Explorer.
  * Navegar a la pestaña **Lineage**, mostrar el grafo aguas arriba hacia `plata.clickstream` y explicar su valor para la auditoría y análisis de impacto.
* **Minuto 4:15 - 5:45 | Automatización con Databricks Workflows / Jobs:**
  * Abrir la sección **Workflows** en Databricks.
  * Mostrar el Job `Wanderbricks_Medallion_Pipeline`, sus dos tareas encadenadas (`01_ingesta_bronce_a_plata` → `02_agregacion_plata_a_oro`) y el historial de corrida exitosa (Succeeded). Explicar por qué se pausó la programación para proteger la cuota de cómputo.
* **Minuto 5:45 - 8:30 | Respuestas a las Tres Preguntas Evaluativas Obligatorias:**
  1. *¿Qué parte de esta arquitectura administra el proveedor y cuál administran ustedes?*
  2. *Muestre un GRANT que usted ejecutó y explique a quién le está dando qué, y por qué.*
  3. *Si tuvieran que montar esto sobre máquinas virtuales, ¿qué sería lo primero que se les complicaría?*
* **Minuto 8:30 - 9:00 | Conclusiones y Cierre:**
  * Resumen del contraste IaaS vs. PaaS para Wanderbricks y despedida.

---

### 📝 Respuestas Fundamentadas a las Tres Preguntas Obligatorias para el Video

#### Pregunta 1: ¿Qué parte de esta arquitectura administra el proveedor y cuál administran ustedes?
> **Respuesta:**  
> En nuestra arquitectura basada en Databricks sobre nube pública rige el **Modelo de Responsabilidad Compartida**:
> - **El Proveedor (Databricks y Cloud Provider)** administra la capa de infraestructura física, el hipervisor de máquinas virtuales, la resiliencia y cifrado de los buckets de almacenamiento objeto S3/ADLS con durabilidad del 99.999999999%, el escalado elástico de cómputo, los parches de seguridad del sistema operativo Linux, la sintonización interna del motor Spark con Photon y la disponibilidad del plano de control de Unity Catalog.
> - **Nosotros (Equipo de Datos)** administramos la capa lógica y de negocio: el diseño de la arquitectura Medallion (Bronce, Plata, Oro), los contratos de datos formales con `StructType`, las reglas de limpieza y aplanado de JSONs, las políticas de gobierno y permisos RBAC (`GRANT` y `REVOKE`), la orquestación de dependencias en Databricks Jobs y los modelos analíticos de consumo para BI.

#### Pregunta 2: Muestre un GRANT que usted ejecutó y explique a quién le está dando qué, y por qué.
> **Respuesta:**  
> *(En el video se enfoca la pantalla en la celda de SQL y se señala la sentencia:)*  
> `GRANT USE SCHEMA, SELECT ON SCHEMA wanderbricks_lakehouse.oro TO \`account users\`;`  
> - **A quién se lo di:** Al grupo de analistas y usuarios de negocio (`account users`).
> - **Qué le otorgué:** Privilegios de `USAGE` sobre el esquema y `SELECT` exclusivamente sobre las tablas agregadas de la capa `oro`.
> - **Por qué:** Por estricto cumplimiento del **Principio de Mínimo Privilegio (PoLP)** y normativas de privacidad (GDPR/Habeas Data). Los analistas de negocio requieren consumir métricas consolidadas (tasa de conversión por dispositivo, ingresos agregados) para tomar decisiones en dashboards de PowerBI o Databricks SQL. No tienen ninguna necesidad de consultar los datos crudos de las capas `bronce` o `plata`, donde residen datos personales como correos electrónicos, nombres, identificadores de pago o payloads completos de clickstream. Al restringir el `GRANT` solo a `oro`, protegemos la superficie de ataque y prevenimos la exposición accidental de PII.

#### Pregunta 3: Si tuvieran que montar esto sobre máquinas virtuales, ¿qué sería lo primero que se les complicaría?
> **Respuesta:**  
> Lo primero y más crítico que se nos complicaría sería **el aprovisionamiento, integración y sincronización de alta disponibilidad del metastore distribuido (Hive Metastore / PostgreSQL) junto con la gestión del estado y fallos del cluster Spark**.  
> En Databricks, Unity Catalog y el catálogo de tablas funcionan de forma transparente y serverless con un solo comando SQL. Si tuviéramos que montarlo sobre IaaS:
> 1. Tendríamos que desplegar y endurecer un motor relacional externo (PostgreSQL), inicializar los esquemas de Hive Metastore versión 3.x, lidiar con incompatibilidades de librerías JAR de Hadoop (`hadoop-aws`, `aws-java-sdk-bundle`), y configurar el servicio Thrift escuchando en el puerto 9083 con llaves Kerberos o SSL.
> 2. Cualquier caída de red entre los workers de Spark y la base del metastore provocaría que el cluster quede 'ciego', impidiendo resolver tablas y esquemas.
> 3. Además, perderíamos el auto-healing: si un nodo worker se queda sin memoria por un *Garbage Collection pause* o un fallo de hardware en IaaS, tendríamos que intervenir manualmente para aprovisionar otra VM, reconfigurar IPs en los archivos `workers/slaves` y relanzar los servicios, mientras que en PaaS el plano de control reemplaza el nodo caído de manera 100% invisible para el usuario.